In [ ]:
!pip install autogluon.tabular -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 6.0 MB/s eta 0:00:00


###250

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#!pip install autogluon.tabular -q

import pandas as pd, numpy as np
import lightgbm as lgb
from autogluon.tabular import TabularPredictor

FEAT_DIR = '/content/drive/MyDrive/ML competition/feature'
DATA_DIR = '/content/drive/MyDrive/ML competition/competition_data'

# === 523 피쳐 구성 (train+test) ===
feats_1st = pd.read_parquet(f'{FEAT_DIR}/feats_1st.parquet')          # 431
v3        = pd.read_parquet(f'{FEAT_DIR}/tx_stack_v3.parquet')        # 24
ex_tr = pd.read_parquet(f'{FEAT_DIR}/extra_features_train.parquet')
ex_te = pd.read_parquet(f'{FEAT_DIR}/extra_features_test.parquet')
tm_tr = pd.read_parquet(f'{FEAT_DIR}/timing_train.parquet')
tm_te = pd.read_parquet(f'{FEAT_DIR}/timing_test.parquet')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')

extra = pd.concat([ex_tr, ex_te]); extra.columns = ['ex_'+c for c in extra.columns]
timing = pd.concat([tm_tr, tm_te])

X = feats_1st.join(v3).join(extra).join(timing)
X.columns = [str(c) for c in X.columns]
X = X.loc[:, ~X.columns.duplicated()].fillna(0)
print('전체 피쳐:', X.shape[1])

tr_ids = y_train['custid'].values
test_ids = [c for c in X.index if c not in set(tr_ids)]
t = y_train.set_index('custid')['gender'].loc[tr_ids].values

# === LightGBM gain importance로 top300 선택 ===
Xtr = X.loc[tr_ids].values.astype(np.float32)
imp = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.04, num_leaves=31,
    subsample=0.8, colsample_bytree=0.55, reg_lambda=2.0, n_jobs=-1, max_bin=127,
    verbose=-1, importance_type='gain').fit(Xtr, t).feature_importances_
feat = list(X.columns)
ranked = [feat[i] for i in np.argsort(-imp)]
TOP = 250
sel = ranked[:TOP]
print(f'선택: top{TOP}')

# === AutoGluon (선택된 피쳐만) ===
Xsel = X[sel]
train_data = Xsel.loc[tr_ids].copy()
train_data['gender'] = t
X_test = Xsel.loc[test_ids]

predictor = TabularPredictor(label='gender', problem_type='binary', eval_metric='roc_auc').fit(
    train_data, presets='best_quality', time_limit=3600)

pred = predictor.predict_proba(X_test)[1]
pd.DataFrame({'custid': X_test.index, 'gender': pred}).to_csv(
    f'{FEAT_DIR}/submission_top250.csv', index=False)
print('AG OOF:', predictor.leaderboard(silent=True).iloc[0]['score_val'])
print('saved submission_top250.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 피쳐: 523


No path specified. Models will be saved in: "AutogluonModels/ag-20260607_071540"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       8.64 GB / 12.67 GB (68.2%)
Disk Space Avail:   85.25 GB / 107.72 GB (79.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stack

선택: top250


Beginning AutoGluon training ... Time limit = 900s
AutoGluon will save models to "/content/AutogluonModels/ag-20260607_071540/ds_sub_fit/sub_fit_ho"
Train Data Rows:    26666
Train Data Columns: 250
Label Column:       gender
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    8864.90 MB
	Train Data (Original)  Memory Usage: 50.76 MB (0.6% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Or

[1000]	valid_set's binary_logloss: 0.539822
[1000]	valid_set's binary_logloss: 0.543852


	0.7214	 = Validation score   (roc_auc)
	195.71s	 = Training   runtime
	1.61s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 1066.44s of the 1066.43s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
No improvement since epoch 2: early stopping
No improvement since epoch 3: early stopping
No improvement since epoch 0: early stopping
No improvement since epoch 5: early stopping
No improvement since epoch 1: early stopping
No improvement since epoch 7: early stopping
No improvement since epoch 2: early stopping
No improvement since epoch 1: early stopping
	0.6962	 = Validation score   (roc_auc)
	499.63s	 = Training   runtime
	0.82s	 = Validation runtime
Fitting model: CatBoost_r9_BAG_L1 ... Training model for up to 565.47s of the 565.46s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: c

[1000]	valid_set's binary_logloss: 0.539032
[1000]	valid_set's binary_logloss: 0.541704
[1000]	valid_set's binary_logloss: 0.534994
[1000]	valid_set's binary_logloss: 0.542992
[1000]	valid_set's binary_logloss: 0.544227
[2000]	valid_set's binary_logloss: 0.543226
[1000]	valid_set's binary_logloss: 0.541153
[1000]	valid_set's binary_logloss: 0.541893
[2000]	valid_set's binary_logloss: 0.540947
[1000]	valid_set's binary_logloss: 0.545981
[2000]	valid_set's binary_logloss: 0.545249


	0.7254	 = Validation score   (roc_auc)
	202.86s	 = Training   runtime
	2.64s	 = Validation runtime
Fitting model: NeuralNetTorch_r22_BAG_L1 ... Training model for up to 358.20s of the 358.19s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
	Ran out of time, stopping training early. (Stopping on epoch 25)
	Ran out of time, stopping training early. (Stopping on epoch 27)
	0.7201	 = Validation score   (roc_auc)
	299.55s	 = Training   runtime
	1.5s	 = Validation runtime
Fitting model: XGBoost_r33_BAG_L1 ... Training model for up to 56.77s of the 56.76s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
	0.6984	 = Validation score   (roc_auc)
	54.95s	 = Training   runtime
	0.24s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 1.16s of remaining time.
	Fitting 1 model 

AG OOF: 0.7277566816025682
saved submission_top300.csv


In [ ]:
# 1. 저장할 경로 설정
save_path = f'{FEAT_DIR}/features_combined_250.parquet'

# 2. Parquet 파일로 저장
# 압축 알고리즘(snappy)을 사용하여 용량과 속도를 모두 최적화합니다.
X.to_parquet(save_path, index=True, compression='snappy')

print(f"🎉 모든 피쳐({X.shape[1]}개)가 파일로 저장되었습니다!")
print(f"📁 경로: {save_path}")

🎉 모든 피쳐(523개)가 파일로 저장되었습니다!
📁 경로: /content/drive/MyDrive/ML competition/feature/features_combined_250.parquet


##225

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#!pip install autogluon.tabular -q

import pandas as pd, numpy as np
import lightgbm as lgb
from autogluon.tabular import TabularPredictor

FEAT_DIR = '/content/drive/MyDrive/ML competition/feature'
DATA_DIR = '/content/drive/MyDrive/ML competition/competition_data'

# === 523 피쳐 구성 (train+test) ===
feats_1st = pd.read_parquet(f'{FEAT_DIR}/feats_1st.parquet')          # 431
v3        = pd.read_parquet(f'{FEAT_DIR}/tx_stack_v3.parquet')        # 24
ex_tr = pd.read_parquet(f'{FEAT_DIR}/extra_features_train.parquet')
ex_te = pd.read_parquet(f'{FEAT_DIR}/extra_features_test.parquet')
tm_tr = pd.read_parquet(f'{FEAT_DIR}/timing_train.parquet')
tm_te = pd.read_parquet(f'{FEAT_DIR}/timing_test.parquet')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')

extra = pd.concat([ex_tr, ex_te]); extra.columns = ['ex_'+c for c in extra.columns]
timing = pd.concat([tm_tr, tm_te])

X = feats_1st.join(v3).join(extra).join(timing)
X.columns = [str(c) for c in X.columns]
X = X.loc[:, ~X.columns.duplicated()].fillna(0)
print('전체 피쳐:', X.shape[1])

tr_ids = y_train['custid'].values
test_ids = [c for c in X.index if c not in set(tr_ids)]
t = y_train.set_index('custid')['gender'].loc[tr_ids].values

# === LightGBM gain importance로 top300 선택 ===
Xtr = X.loc[tr_ids].values.astype(np.float32)
imp = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.04, num_leaves=31,
    subsample=0.8, colsample_bytree=0.55, reg_lambda=2.0, n_jobs=-1, max_bin=127,
    verbose=-1, importance_type='gain').fit(Xtr, t).feature_importances_
feat = list(X.columns)
ranked = [feat[i] for i in np.argsort(-imp)]
TOP = 225
sel = ranked[:TOP]
print(f'선택: top{TOP}')

# === AutoGluon (선택된 피쳐만) ===
Xsel = X[sel]
train_data = Xsel.loc[tr_ids].copy()
train_data['gender'] = t
X_test = Xsel.loc[test_ids]

predictor = TabularPredictor(label='gender', problem_type='binary', eval_metric='roc_auc').fit(
    train_data, presets='best_quality', time_limit=3600)

pred = predictor.predict_proba(X_test)[1]
pd.DataFrame({'custid': X_test.index, 'gender': pred}).to_csv(
    f'{FEAT_DIR}/submission_top225.csv', index=False)
print('AG OOF:', predictor.leaderboard(silent=True).iloc[0]['score_val'])
print('saved submission_top225.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 피쳐: 523


No path specified. Models will be saved in: "AutogluonModels/ag-20260607_083829"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       8.78 GB / 12.67 GB (69.3%)
Disk Space Avail:   84.18 GB / 107.72 GB (78.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stack

선택: top225


Beginning AutoGluon training ... Time limit = 900s
AutoGluon will save models to "/content/AutogluonModels/ag-20260607_083829/ds_sub_fit/sub_fit_ho"
Train Data Rows:    26666
Train Data Columns: 225
Label Column:       gender
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    9025.86 MB
	Train Data (Original)  Memory Usage: 45.67 MB (0.5% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Or

[1000]	valid_set's binary_logloss: 0.54059
[1000]	valid_set's binary_logloss: 0.544733


	0.7214	 = Validation score   (roc_auc)
	198.78s	 = Training   runtime
	1.76s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 1076.87s of the 1076.83s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
No improvement since epoch 7: early stopping
No improvement since epoch 7: early stopping
No improvement since epoch 2: early stopping
No improvement since epoch 5: early stopping
No improvement since epoch 1: early stopping
No improvement since epoch 2: early stopping
No improvement since epoch 3: early stopping
No improvement since epoch 1: early stopping
	0.6993	 = Validation score   (roc_auc)
	516.01s	 = Training   runtime
	0.83s	 = Validation runtime
Fitting model: CatBoost_r9_BAG_L1 ... Training model for up to 559.50s of the 559.46s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: c

[1000]	valid_set's binary_logloss: 0.538492
[1000]	valid_set's binary_logloss: 0.541984
[1000]	valid_set's binary_logloss: 0.53475
[1000]	valid_set's binary_logloss: 0.543342
[1000]	valid_set's binary_logloss: 0.5442
[1000]	valid_set's binary_logloss: 0.540003
[1000]	valid_set's binary_logloss: 0.542233
[1000]	valid_set's binary_logloss: 0.545352


	0.7252	 = Validation score   (roc_auc)
	191.01s	 = Training   runtime
	2.27s	 = Validation runtime
Fitting model: NeuralNetTorch_r22_BAG_L1 ... Training model for up to 364.58s of the 364.54s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
	Ran out of time, stopping training early. (Stopping on epoch 25)
	Ran out of time, stopping training early. (Stopping on epoch 24)
	0.7211	 = Validation score   (roc_auc)
	304.39s	 = Training   runtime
	1.63s	 = Validation runtime
Fitting model: XGBoost_r33_BAG_L1 ... Training model for up to 58.08s of the 58.04s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
	0.6895	 = Validation score   (roc_auc)
	55.41s	 = Training   runtime
	0.35s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 1.77s of remaining time.
	Fitting 1 model

AG OOF: 0.7277000803289561
saved submission_top225.csv


In [ ]:
# ranked가 메모리에 있는 상태에서 (방금 돌린 셀의 변수 재사용)
for TOP in [250, 300, 225]:
    sel = ranked[:TOP]
    Xsel = X[sel]                       # train+test 합쳐진 X
    Xsel.to_parquet(f'{FEAT_DIR}/feats_top{TOP}.parquet')
    print(f'저장: feats_top{TOP}.parquet', Xsel.shape)

저장: feats_top250.parquet (49995, 250)
저장: feats_top300.parquet (49995, 300)
저장: feats_top225.parquet (49995, 225)
